In [1]:
#Install Packages
!pip install faiss-cpu
!pip install sentence-transformers


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# import necessary libraries
import pandas as pd
pd.set_option('display.max_colwidth', 100)

In [20]:
df = pd.read_csv(r"C:\Users\mr\Downloads\sample_text.csv")
df.shape

(8, 2)

In [5]:
df

,text,category
0,Meditation and yoga can improve mental health,Health
1,"Fruits, whole grains and vegetables helps control blood pressure",Health
2,These are the latest fashion trends for this week,Fashion
3,Vibrant color jeans for male are becoming a trend,Fashion
4,The concert starts at 7 PM tonight,Event
5,Navaratri dandiya program at Expo center in Mumbai this october,Event
6,Exciting vacation destinations for your next trip,Travel
7,Maldives and Srilanka are gaining popularity in terms of low budget vacation places,Travel


In [25]:
from sentence_transformers import SentenceTransformer

In [26]:
encoder = SentenceTransformer("all-mpnet-base-v2")
vectors = encoder.encode(df.text)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2087.69it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [27]:
vectors.shape

(8, 768)

In [28]:
dim = vectors.shape[1]
dim

768

In [29]:
import faiss

index = faiss.IndexFlatL2(dim)

In [30]:
index.add(vectors)

In [31]:
index

<faiss.swigfaiss_avx2.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x000001A980C3BC00> >

In [32]:
search_query = "I want to buy a polo t-shirt"
# search_query = "looking for places to visit during the holidays"
# search_query = "An apple a day keeps the doctor away"
vec = encoder.encode(search_query)
vec.shape

(768,)

In [33]:
import numpy as np
svec = np.array(vec).reshape(1,-1)
svec.shape

(1, 768)

In [34]:
faiss.normalize_L2(svec)

In [36]:
import faiss

index = faiss.IndexFlatL2(dim)

In [37]:
index.add(vectors)

In [38]:
vectors

array([[-0.00247393,  0.03626724, -0.05290457, ..., -0.09152355,
        -0.03970002, -0.04330488],
       [-0.03357266,  0.00980519, -0.03250128, ..., -0.05165466,
         0.02245887, -0.03156182],
       [-0.01865324, -0.04051313, -0.01235388, ...,  0.00610587,
        -0.07179645,  0.02773851],
       ...,
       [-0.00066459,  0.04252128, -0.05645507, ...,  0.01315472,
        -0.03183567, -0.04357663],
       [-0.03317154,  0.03252457, -0.02484838, ...,  0.01174421,
         0.05747123,  0.00571022],
       [-0.00166394,  0.00413825, -0.04597083, ...,  0.02008527,
         0.05656242, -0.00161596]], shape=(8, 768), dtype=float32)

In [39]:
search_query = "I want to buy a polo t-shirt"
# search_query = "looking for places to visit during the holidays"
# search_query = "An apple a day keeps the doctor away"
vec = encoder.encode(search_query)
vec.shape

(768,)

In [45]:
# Step 5: Search for similar vector in the FAISS index
distances, I = index.search(svec, k=2)  # ← svec, not new_vec

print("Distances:", distances)
print("Indices:", I)

# Step 6: Retrieve the matching results from your dataframe
print("\nTop matching results:\n")
for rank, idx in enumerate(I[0]):
    print(f"--- Result {rank+1} (distance: {distances[0][rank]:.4f}) ---")
    print(df.text.iloc[idx])
    print()

Distances: [[1.3844836 1.4039094]]
Indices: [[3 2]]

Top matching results:

--- Result 1 (distance: 1.3845) ---
Vibrant color jeans for male are becoming a trend

--- Result 2 (distance: 1.4039) ---
These are the latest fashion trends for this week



In [46]:
I

array([[3, 2]])

In [47]:
I.tolist()

[[3, 2]]

In [48]:
row_indices = I.tolist()[0]
row_indices

[3, 2]

In [49]:
df.loc[row_indices]

,text,category
3,Vibrant color jeans for male are becoming a trend,Fashion
2,These are the latest fashion trends for this week,Fashion


In [50]:
search_query

'I want to buy a polo t-shirt'